So far we discussed supervised learning techniques for regression and classification tasks, i.e. predicting a continuous value or a discrete label. In this section, we will explore unsupervised learning techniques, i.e. methods that learn patterns and informations from unlabeled data.

# Association Rules Mining
It refers to a set of techniques used to discover interesting relationships, patterns, or associations among a set of items in large datasets. It is commonly used in market basket analysis, where the goal is to identify items that frequently co-occur in transactions.

So, the objective of association rules mining is the extraction of frequent correlations or patterns from a **transactional database** (a collection of transactions, where each transaction is a set of items purchased together, ex. 1. {milk, bread, eggs}, 2. {bread, butter}, 3. {milk, butter} etc.).

Given a collection of transactions, where a transaction is a **not ordered** set of items, an association rule is an implication of the form:
$ A, B \rightarrow C $  
A, B are items in the rule body, while C is the item in the rule head. The rule can be read as "if A and B are purchased together, then C is also likely to be purchased".

This can be done also with **Text Data**, where we consider documents as transactions and words as items. The goal is to find associations between words that frequently co-occur in documents. 

We can do the same with **Structured Data**, i.e. data organized in rows and columns (like a database table or a spreadsheet). Here the idea is to consider the pair (attribute, value) as an item. For example, in a dataset with attributes "Color" and "Size", the items could be ("Color", "Red") and ("Size", "Large"). The goal is to find associations between different attribute-value pairs that frequently occur together in the dataset. ex. {("Refund", "No"), ("Married", "Yes")} $\rightarrow$ {("Cheat", "No")}

Notice in general that we are not predicting a specific target variable, but rather discovering interesting patterns and relationships within the data itself (we are not saying "a person will buy item C if they buy A and B", but rather "people who buy A and B often also buy C").
Of course anyway we could use association rules for prediction tasks, but this is not the main goal of the technique.

## Definitions
In order to understand how to find useful association rules, we need to define some important concepts. Imagine to have the following transactions:

<img src="img_teoria/transactions.png" width="300">

Then we can define:
- **Itemset**: A set of items. For example, {Beer, Diapers} is an itemset.
- **k-itemset**: An itemset containing k items.
- **Support Count** (#): The number of transactions that contain a particular itemset. For example, #{Beer, Diapers} = 2, since there are 3 transactions that contain both Beer and Diapers.
- **Support** (s): The proportion of transactions that contain a particular itemset. It is calculated as:
$ sup(A) = \frac{\text{\#}(A)}{N} $  
where #(A) is the support count of itemset A, and N is the total number of transactions. For example, sup({Beer, Diapers}) = 2/5 = 0.4.
- **Frequent Itemset**: An itemset who appears frequently enough, i.e. its support is greater than or equal to a predefined minimum support threshold (minsup). For example, if minsup = 0.4, then {Beer, Diapers} is a frequent itemset.

### Rule Metrics
Given the association rule A $\rightarrow$ B, where A and B are itemsets, we can define some metrics to evaluate the quality of the rule:
- **Support** (s): The proportion of transactions that contain both A and B. It is calculated as:
$ sup(A \rightarrow B) = \frac{\text{\#}(A,B)}{N} $  
where #(A,B) is the support count of the union of itemsets A and B, and N is the total number of transactions. This gives us an idea of how frequently the rule occurs in the dataset.  
ex. if the support of the rule {Beer} $\rightarrow$ {Diapers} is 0.4, it means that 40% of all transactions contain both Beer and Diapers.
- **Confidence** (c): The proportion of transactions that contain A and also contain B (i.e. the frequency of B in transactions that contain A, i.e. the conditional probability of B given A). It is calculated as:
$ conf(A \rightarrow B) = \frac{\text{sup}(A,B)}{\text{sup}(A)} $  
ex. if the confidence of the rule {Diapers} $\rightarrow$ {Beer} is 0.67, it means that in 67% of the transactions where Diapers is purchased, Beer is also purchased.

An example below:

<img src="img_teoria/rule_metrics.png" width="400">

## Association Rule Extraction
The task is the following. Given a set of transactions T, association rule mining is the extraction of rules satisfying the following constraints:
- support $\geq$ minsup threshold
- confidence $\geq$ minconf threshold

The goal is to find a result that is:
- **Complete**: all rules satisfying the minsup and minconf thresholds are found
- **Correct**: all found rules satisfy the minsup and minconf thresholds

In order to accomplish this task, the first idea might be to use a **brute-force approach**: generate all possible rules and evaluate their support and confidence, prune the rules that don't satisfy the minsup and minconf thresholds. This approach is computationally expensive and not feasible for large datasets.

Another idea might be to first generate only **frequent itemsets** (i.e. itemsets with support $\geq$ minsup), and then generate rules from these frequent itemsets. It is still exponential,since generating all possible rules for an itemset of size k results in 2^k - 2 possible rules (ex. for {Milk, Diapers} we can have Milk $\rightarrow$ Diapers and Diapers $\rightarrow$ Milk), but we would reduce significantly the number of itemsets to consider (could be feasible).

So we want to approach the second idea, and the first question is **how to efficiently find frequent itemsets?**

Given d items, the total number of possible itemsets is 2^d - 1 (excluding the empty set). This is clearly infeasible for large d.  

<img src="img_teoria/letex.png" width="400">

With brute force to find frequent itemsets we would need to consider all possible itemsets, compute the support of each one and check if it is greater than minsup. Counting the support requires scanning the entire dataset, so the overall complexity is O(|T| * 2^d * w), where |T| is the number of transactions and w is the average transaction width (number of items in a transaction). This is clearly unfeasible for large d, because of the exponential term 2^d.

We could reduce the **number of candidate itemsets**, pruning the search space; reduce the **number of transactions to scan** and reduce the **number of comparisons** needed to count the support.

### Apriori Algorithm
How to generate frequent itemsets efficiently? The most famous algorithm is the **Apriori Algorithm**, which uses the **Apriori Principle** to prune the search space of candidate itemsets.

The **Apriori Principle** states that **"if an itemset is frequent, then all of its subsets must also be frequent"**. Ex. if {Milk, Diapers} is frequent, then {Milk} and {Diapers} must also be frequent, in fact if 1000 people buy milk and diapers, at least 1000 must buy milk and at least 1000 must buy diapers (at least because more people could buy only one of them). 

**This means that if we find an itemset that is not frequent, we can prune all its supersets from consideration.** In fact, saying that if an itemset is frequent then all its subsets must also be frequent is equivalent to saying that if an itemset is not frequent then all its supersets cannot be frequent (ex. if {Milk} is not frequent, then {Milk, Diapers} cannot be frequent).

L'apriori algorithm prevede di estrarre, ad ogni iterazione, gli itemset frequenti di lunghezza k (k-itemset) a partire dagli itemset frequenti di lunghezza k-1. 

L'algoritmo procede in questo modo, per ogni iterazione k = 0, 1, 2, ... finché non si trovano più itemset frequenti:
1. **Candidate Generation**:
- Join Step: genera i candidati k+1-itemset unendo gli itemset frequenti di lunghezza k.
- Prune Step: si applica il principio di Apriori: se un candidato k+1-itemset ha un sottoinsieme di lunghezza k che non è frequente, allora il candidato non può essere frequente e viene eliminato.
2. **Frequent Itemset Generation**:
- Si scansiona il database delle transazioni per contare il supporto di ciascun candidato k+1-itemset.
- Si eliminano i candidati che non soddisfano la soglia di supporto minima (minsup), ottenendo così gli itemset frequenti di lunghezza k+1.

Ma come generare i candidati k+1-itemset dai k-itemset frequenti? Sia $L_{k}$ l'insieme degli itemset frequenti di lunghezza k. Per prima cosa, $L_{k}$ viene ordinato in ordine lessicografico. Dopodiché, si effettua una self-join per ogni candidato che abbia i primi k-1 elementi in comune. Una volta ottenuti i candidati, si applica il prune step del principio di Apriori per eliminare i candidati che hanno sottoinsiemi non frequenti. (con self-join si intende unire l'insieme con se stesso, es. abc e abd hanno in comune i primi 2 elementi ab, quindi si uniscono per formare abcd).

esempio: L_3 = {abc, abd, acd, ace, bcd}
- join step: C_4 = {abcd, acde} (abcd da abc e abd, acde da acd e ace)
- prune step: eliminiamo acde perché ha almeno un sottinsieme non frequente (Apriori principle, ade, cde non sono in L_3, se almeno un sottinsieme non è frequente, allora anche il superset non può esserlo), quindi C_4 = {abcd}.

Let's see an example of the Apriori algorithm in action. Given the transactions below and minsup = 1 (i.e. an itemset is frequent if it appears in at least 1 transaction), the first step is to find the frequent 1-itemsets. In order to do this, the only way for the first iteration is to scan the entire dataset and count the support of each item. We obtain a set C1, and since all the candidates have support $\geq$ minsup, we have L1 = C1. 

<img src="img_teoria/apriori_1.png" width="500">

At the second iteration, we generate candidate 2-itemsets from L1 using the join and prune steps. In questo caso, k-1 = 0 -> si effettua una self-join per ogni coppia di itemset che hanno i primi 0 elementi in comune, quindi si uniscono tutti gli itemset. Si ottiene quindi un insieme C2 contenente tutte le possibili combinazioni di 2 item dall'insieme L1.  Questa iterazione è un punto critico per Apriori, dal momento che possiamo generare un numero quadratico di candidati (se abbiamo m itemset in L1, possiamo generare fino a m*(m-1)/2 candidati in C2). 

Facciamo quindi ancora il pruning, tuttavia tutti i candidati in L1 erano frequenti, quindi anche tutti i candidati in C2 lo sono (per il principio di Apriori). Si effettua quindi una scansione del database per contare il supporto di ciascun candidato in C2, e si eliminano quelli che non soddisfano minsup, ottenendo L2.

<img src="img_teoria/apriori_2.png" width="500">

Si passa quindi alla terza iterazione. Stavolta k-1 = 1, quindi si effettua una self-join per ogni coppia di itemset in L2 che hanno solo il primo elemento in comune. Si ottiene quindi C3. Si effettua il prune step: ad esempio nel nostro caso {A,B,E} contiene {B,E} che non fa parte di L2 (quindi non è frequente) -> rimuovo {A,B,E} da C3 per l'apriori principle. Si effettua quindi una scansione del database per contare il supporto di ciascun candidato in C3, e si eliminano quelli che non soddisfano minsup, ottenendo L3.

<img src="img_teoria/apriori_3.png" width="500">

Si continua così fino a quando non si trovano più itemset frequenti. In questo caso, alla quarta iterazione non si trovano candidati frequenti, quindi l'algoritmo termina.

<img src="img_teoria/apriori_4.png" width="500">

In questo esempio, abbiamo estratto tutti gli itemset frequenti di lunghezza 1, 2 e 3, effettuando un totale di 4 scansioni del database (una per ogni iterazione, considerando anche la prima scansione iterazione 0 per trovare gli itemset di lunghezza 1).
**Più in generale,  per ottenere gli itemset più frequenti fino a lunghezza k, è necessario effettuare almeno k scansioni del database.**

L'algoritmo Apriori quindi sicuramente è nettamente migliore del brute-force, ma ci sono alcuni problemi:
- 2-itemset candidate generation è il most critical step, perché può generare un numero quadratico di candidati
- per trovare itemset di lunghezza k, bisogna prima trovare gli itemset di lunghezza inferiore (k-1, k-2, ...), quindi se vogliamo itemset lunghi dobbiamo fare molte iterazioni
- se vogliamo itemset di lunghezza n, dobbiamo fare n+1 = O(n) scansioni del database, che può essere costoso se il database è grande.

Altri fattori che influenzano le prestazioni dell'algoritmo Apriori sono:
- **La scelta della soglia di supporto minima** (minsup): una soglia più alta riduce il numero di itemset frequenti, ma potrebbe far perdere regole interessanti. Una soglia più bassa aumenta il numero di itemset frequenti, ma può portare a un aumento del tempo di calcolo e a un maggior numero di regole da analizzare.
- **Dimensionality**: it refers to the number of unique items in the dataset. A higher dimensionality increases the number of possible itemsets, and in Apriori this can be problematic especially for the 2-itemset candidate generation step
- **Database Size**: we must perform a database scan for each iteration of the algorithm, so larger databases will increase the overall computation time.
- **Avg transaction width**: wider (dense) transactions contain more items, which can lead to a larger number of candidate itemsets being generated and evaluated, increasing the computational complexity of the algorithm.

In literature, there have been proposed many improvements to the Apriori algorithm to address these issues, one of them is the **FP-Growth algorithm**, which uses a different approach based on a compact data structure called FP-tree to mine frequent itemsets without candidate generation, significantly improving efficiency.

### FP-Growth Algorithm
The idea behind the FP-Growth algorithm is to address the problems of the Apriori algorithm about scanning the database multiple times and having to deal with dense transactions, we want an algorithm that scans the database a constant number of times and builds a data structure that compresses the database.

The FP-Tree represents a data structure that compresses the database by storing only the frequent items and their counts in a tree structure. This way, we have high compression in particular for dense transactions -> it "benefits" from having large and dense transactions.

After having built the FP-Tree, we do the so called **frequent pattern mining**, which consists in recursively extracting frequent itemsets from the FP-Tree without generating candidates. This is done by constructing conditional FP-Trees for each frequent item and recursively mining them to find all frequent itemsets. We'll see an example to clarify the process.

The most interesting thing about FP-Growth is that it requires only 2 scans of the database to build the FP-Tree, regardless of the number of frequent itemsets or their lengths. This is a significant improvement over Apriori, especially for large datasets with long frequent itemsets. *The fist scan is to count the support of each item and determine the frequent items, while the second scan is to build the FP-Tree by inserting transactions in the tree structure based on the frequent items.*

Let's see an example of how FP-Growth works. Per prima cosa, dato il database con le transazioni, facciamo il primo scan del database per contare il supporto di ciascun item e determinare gli item frequenti (sopra la soglia minsup). In questo modo costruiamo la **Header Table**, che contiene gli item frequenti **ordinati in ordine decrescente di supporto**.

<img src="img_teoria/fp_1.png" width="400">

After that, we create the FP-Tree by following these steps:
- order transaction items based on the Header Table order (items in each transaction are ordered according to their frequency in the Header Table)
- insert each ordered transaction into the FP-Tree, updating counts and creating new nodes as necessary

<img src="img_teoria/fp_2.png" width="400">
<img src="img_teoria/fp_3.png" width="400">
<img src="img_teoria/fp_4.png" width="400">
<img src="img_teoria/fp_5.png" width="400">

After all 10 transactions have been inserted, we obtain the final FP-Tree:

<img src="img_teoria/fp_6.png" width="400">

The nice thing about the FP-Tree is that, if we look for example at the path starting from the leaf D-C-B (terzo da sinistra), dato che D counts 1 -> possiamo concludere che la combinazione D-C-B appare 1 volta nel database, senza doverlo scansionare. In questo modo, il FP-Tree comprime il database e ci permette di estrarre informazioni senza doverlo scansionare più volte.

The final thing that we keep track of, useful for computational efficiency, is the **node-link structure**: for each item in the Header Table, we maintain a linked list of all nodes in the FP-Tree that contain that item. This allows us to quickly access all occurrences of an item in the tree during the mining process.

<img src="img_teoria/fp_7.png" width="400">

Now what we do to mine the frequent itemset is look at the header table from the bottom to the top (starting from the least frequent item). In our case, it's E. At this step, we'll extract **all frequent itemsets that include the item E**. After this iteration we'll move to the next item in the header table (D) and extract all frequent itemsets that include D (that won't include E of course since they were already extracted), and so on until we reach the top of the header table.

The general passes are the following:
- Scan the header table from the bottom to the top
- for each item i in the header table:
  - construct the **Conditional Pattern Base** for i (the set of prefix paths in the FP-Tree that lead to i)
  - build the **Conditional FP-Tree** for i from the conditional pattern base
  - recursively mine the conditional FP-Tree to find all frequent itemsets that include i

  Let's follow the example for D. First we ask if D is frequent alone. If no, we skip it and move to the next item in the header table. If yes, we add D to the current frequent itemsets.

  Now, we construct the conditional pattern base for D. To do this, we follow the node-link structure for D in the FP-Tree, and for each occurrence of D, we trace back to the root to find the prefix path. We also keep track of the count of D at each occurrence. We get the *D-CPB* that is the following:

<img src="img_teoria/fp_8.png" width="400">

Now, with this new dataset (**much smaller than the original one, scanning it isn't as expensive**), we recursively apply the FP-Growth algorithm to find all frequent itemsets that include D. So we compute the D-conditional Header Table *D-CHT*, containing items that are frequent in the D-CPB and order them by support:

<img src="img_teoria/fp_9.png" width="400">

Now we reapply the FP-Growth steps to build the D-Conditional FP-Tree:

<img src="img_teoria/fp_10.png" width="400">

Now, we recursively mine the D-Conditional FP-Tree by invocating FP-Growth on it. We start from the bottom of the D-CHT, which is C. We check if C is frequent, if yes we add DC to the current frequent itemsets (which already contains D) -> we add DC to the current frequent itemsets. Then we build the DC-CPB:

<img src="img_teoria/fp_11.png" width="400">

So we consider DC-CPB as the new dataset, and we build the DC-CHT and the DC-Conditional FP-Tree:

<img src="img_teoria/fp_12.png" width="400">

Now we recursively mine the DC-Conditional FP-Tree. Starting from the bottom of the DC-CHT, which is B. We check if B is frequent, if yes we add DCB to the current frequent itemsets -> we add DCB to the current frequent itemsets. Then we build the DCB-CPB:

<img src="img_teoria/fp_13.png" width="400">

Which contains only A with count 1. Now we count again, ci accorgiamo che A non è frequente (DCBA non è frequente), quindi non aggiungiamo altro ai frequent itemsets. Abbiamo finito di estrarre gli itemset che contengono DCB, quindi torniamo indietro al DC-conditional FP-Tree e passiamo al prossimo item nell'header table, che è A.

<img src="img_teoria/fp_14.png" width="400">

etc...

Dopo aver fatto tutti questi passaggi, abbiamo ottenuto tutti gli itemset frequenti che contengono D. Si tornerà poi all'FP-Tree originale e si passerà al prossimo item nell'header table, che è C, e si ripeterà il processo per estrarre tutti gli itemset frequenti che contengono C (ma non D, perché già estratti). Il tutto continuerà fino a quando non si raggiunge la cima dell'header table, estraendo così tutti gli itemset frequenti dal database originale.

There are many other approaches to frequent item extraction, some of them consider data in a vertical perspective (instead of considering the series of transactions, we consider each item and in which transaction it appears), that permits to create new data structures that can be more efficient in certain cases.

<img src="img_teoria/vertical_data.png" width="400">

All the algorithms that we discussed give us all and only the frequent itemsets, the problem is that **these solutions can be a lot**. For example, below we have a very limited dataset of 15 transactions and around 30 items, and we have a 1 if an item belongs to a transaction, 0 otherwise. Now imagine that our threshold is 5 (an item must appear in at least 5 transactions to be considered frequent).

Just for the first 10 items, we have in total sum(k=1 to 10) (10 choose k) = 2^10 - 1 = 1023 possible itemsets, but in this case the only itemsets that are actually meaningful are probably {A1, ..., A10}, {B1, ..., B10}, {C1, ..., C10}, not all their possible subsets. So these subsets are redundant information that we don't need.

<img src="img_teoria/redundant_itemsets.png" width="400">

So we need to represent the frequent itemsets in a more compact way, removing redundant information. There are many approaches to do this, we will see two of them: **Maximal Frequent Itemsets** and **Closed Frequent Itemsets**.
- **Maximal Frequent Itemsets**:

We define a **Maximal Frequent Itemset** as a frequent itemset that is not a subset of any other frequent itemset. In other words, it is a frequent itemset that cannot be extended by adding more items without losing its frequency. So in the example {BC} is frequent, but it is probably not interesting because it is a subset of the larger frequent itemset {BCD}...

<img src="img_teoria/maximal_itemsets.png" width="400">

So in our case we could compact all the (white) frequent itemsets into the (blue) maximal frequent itemsets {AD}, {ACE}, {BCDE}, and if I store only these three I can reconstruct all the frequent itemsets by generating all their subsets.

### Compact representation of frequent itemsets

- **Closed Itemsets**

We define a **Closed Itemset** as an itemset that has no superset with the same support. In other words, it is a itemset that cannot be extended by adding more items without decreasing its support.

In the example below, {B,D} is a closed itemset because all its supersets ({B,D,C}, {A,B,D}, {A,B,C,D}) have a lower support compared to {B,D}. {C,D} is not closed because its superset {B,C,D} has the same support (so it is more interesting to keep {B,C,D} instead of {C,D}).

Of course the number of closed itemsets is >= than the number of maximal frequent itemsets, because every maximal frequent itemset is also closed (if an itemset is maximal frequent, it cannot have any superset that is frequent, so it cannot have any superset with the same support), but not every closed itemset is maximal frequent (a closed itemset can have supersets that are not frequent).

But by keeping all closed itemsets we can reconstruct all frequent itemsets along with their support values, because for each closed itemset we can generate all its subsets and assign them the same support value as the closed itemset.

(infatti se ho un closed itemset {A,B,C} con supporto 5, so che anche {A,B}, {A,C}, {B,C}, {A}, {B}, {C} hanno supporto 5, perché altrimenti {A,B,C} non sarebbe closed)

Quindi rispetto ai maximal frequent itemsets, i closed itemsets permettono di ricostruire non solo quali sono gli itemset frequenti, ma anche il loro supporto, tuttavia richiedono di memorizzare più itemset.

<img src="img_teoria/closed_itemsets.png" width="400">

- Closed vs Maximal

Below a representation of the relations between frequent itemsets, closed itemsets and maximal frequent itemsets.
In pratica se ho un closed itemset allora tutti i subset hanno supporto uguale, mentre per i maximal frequent itemsets dato che mi dicono solo che il superset non è frequente, potrei avere subset con supporti maggiori. 

<img src="img_teoria/closed_vs_maximal.png" width="400">

**Choosing the right support threshold**:
- If minsup is too high, we may miss interesting patterns (few frequent itemsets) and we may get only obvious rules (such as "people who buy bread also buy milk").
- If minsup is too low, we may get too many frequent itemsets, including many that are not interesting or useful, it could become computationally very expensive.

## Other metrics
So far we only discussed support and confidence as metrics to evaluate the quality of association rules. (dove la confidence indica la probabilità condizionata di B dato A, cioè quanto spesso B viene acquistato quando A è acquistato). However, there are other metrics that can provide additional insights into the strength and interestingness of association rules. We divide metrics in **Objective measures** if they are based on statistics computed from data (like support and confidence), and **Subjective measures** if they depend on user preferences or domain knowledge. For subjective measures, "it is interesting if it contradicts the expectations of the user" (ex. a rule that goes against common sense or domain knowledge might be considered interesting) and "interesting if it is actionable" (ex. a rule that suggests a clear action to take, like "if a customer buys item A, recommend item B").

Why should we consider other metrics besides support and confidence? *Because confidence alone can be misleading*. Let's consider the example that given 5000 students, 3750 of them eat cereals, 3000 play basket and 2000 eat cereals and play basket.
After running our algorithms, we get that the rule {play basket} $\rightarrow$ {eat cereals} has a support of 2000/5000 = 0.4 and a confidence of 2000/3000 = 0.67. This seems to suggest that playing basket is a good predictor for eating cereals. However, if we look at the overall probability of eating cereals, we see that 3750/5000 = 0.75 of students eat cereals regardless of whether they play basket or not. This means that the rule {play basket} $\rightarrow$ {eat cereals} is actually less informative than simply knowing that a student eats cereals in general.

This happens because the head of the rule ({eat cereals}) has high frequency in the dataset, so the confidence of the rule is high simply because eating cereals is common among students, not because playing basket is a good predictor for eating cereals. What we are actually hiding here is that there is actually a **negative correlation** between playing basket and eating cereals (students who play basket are less likely to eat cereals than the general student population), if I only look at confidence I miss this important information. 

In order to solve this we define **Correlation** (or **lift**) of an association rule $r: A \rightarrow B$ :  
$ Correlation(r) = P(A,B)/P(A)P(B) = \frac{conf(r)}{sup(B)} $

If the numerator is greater than the denomenator (the probability that someone eats cereal given that they play basket is greater than the overall probability that someone eats cereal) -> **Correlation > 1** -> **positive correlation**, meaning that playing basket increases the likelihood of eating cereals. If the numerator is less than the denomenator -> **Correlation < 1** -> **negative correlation**, meaning that playing basket decreases the likelihood of eating cereals. If they are equal -> **Correlation == 1** -> **no correlation**.  
If the Correlation = 1 -> no correlation (because P(A,B) = P(A)P(B), meaning that A and B are independent, playing basketball does not affect the likelihood of eating cereals).

For our example, we get that play basket -> eat cereals has a correlation of (2000/5000) / ((3000/5000)*(3750/5000)) = 0.4 / 0.45 = 0.89, which indicates a negative correlation between playing basket and eating cereals.

But if we take play basket -> not (eat cereals) we get a correlation of (1500/5000) / ((3000/5000)*(1250/5000)) = 0.3 / 0.225 = 1.34, which indicates a positive correlation between playing basket and not eating cereals (so in this case this is the interesting rule).

## Considering weights or hierarchies
- **Considering weights**:

We could consider items by different importance within a transaction (ex. some items are more expensive, or more relevant for the business). In this case we can assign a weight to each item, that measures its relevance in the corresponding transaction.

- **Considering hierarchies**:

In our data warehouses we can have multiple dimensions for the same item, organized in hierarchies (ex. item "milk" belongs to category "dairy products", which belongs to category "food", etc.). In this case we can consider association rules at different levels of abstraction (ex. "if a customer buys dairy products, they are likely to buy bread" instead of "if a customer buys milk, they are likely to buy whole wheat bread"). We call an itemset like this a **generalized itemset**, and the corresponding rules are called **generalized association rules**.

Why is it useful? Imagine a transaction where user:John at time:6.05pm uses service:weather. This transaction has very low support (s = 0.0005%), but if we generalize user as employee and time as work hours, the generalized transaction (employee, 6 to 7 pm, service:weather) has a much higher support (s = 0.2%), allowing us to discover more general patterns.